# L17 · 特征分解（eigendecomposition） A=PDP⁻¹ — 换坐标系让矩阵乘法变成标量乘法

**目标**：掌握 `Av = λv`、特征方程（characteristic equation） `det(A − λI) = 0`；对称矩阵（symmetric matrix）可对角化（diagonalization） `A = PΛP⁻¹`，矩阵幂次 `Aⁿ = PΛⁿP⁻¹` 退化为标量幂次。

**为什么对 Aurora 重要**：递归滤波器（recursive filter）和状态空间模型（state space model）的稳定性由系统矩阵的特征值（eigenvalue）决定——所有 `|λ| < 1` 则滤波器收敛；`aurora.audio.transforms` 的极点分析依赖这套分解。

← **上一课**　[L16 · 行列式与逆矩阵](L16_determinant_inverse.ipynb)

> 上节课学习了 **行列式与逆矩阵**：det(A) 的几何含义（面积缩放），求逆与何时不可逆。  
> 本课将探讨 **特征分解 A=PDP⁻¹**。

## 本课剧情：换个坐标系，乘法变成普通数乘

你有没有想过：为什么某些计算在特定坐标系下特别简单？

普通矩阵乘法 `A @ v` 很难——每次都要做 n² 次乘加。但如果 `v` 是特征向量，乘法退化成**标量乘法**：`A @ v = λ · v`，只需乘一个数。

特征分解（eigendecomposition）就是找到一组"天然坐标系"，让矩阵在这个基底下变成对角矩阵（diagonal matrix）D：

```
A = P D P⁻¹
```

含义：
- **P**：换到特征向量坐标系
- **D**：在新坐标系下，A 的作用只是各方向独立缩放（对角元素 = 特征值）
- **P⁻¹**：换回原始坐标系

这个分解有什么用？
- **矩阵幂次**：`Aⁿ = P Dⁿ P⁻¹`，Dⁿ 只需对角线各元素自乘 n 次
- **稳定性分析**：如果所有 |λᵢ| < 1，则 Aⁿ → 0（系统收敛）
- **PCA**：找协方差矩阵的特征向量 = 数据变化最大的方向

本课实现 `char_poly(A, λ) = det(A - λI)`，验证真正的特征值使其为零。

## 1. 特征方程：`det(A − λI) = 0`

**如何找特征值？**

`Av = λv` 等价于 `(A - λI)v = 0`，有非零解 v 当且仅当 `A - λI` 奇异：

$$\det(A - \lambda I) = 0$$

这个方程的根就是特征值。

**2×2 手算例子**：A = [[4, 1], [2, 3]]

```
det(A - λI) = (4-λ)(3-λ) - 2·1
            = λ² - 7λ + 12 - 2
            = λ² - 7λ + 10
            = (λ-5)(λ-2) = 0
```

特征值：λ₁ = 5，λ₂ = 2。

**Python 验证**：`np.linalg.eig(A)` 直接返回特征值，`char_poly(A, 5)` 应约等于 0。

> n×n 矩阵有 n 个特征值（含重复和复数）；实对称矩阵的特征值全为实数。

## 符号入口：先看形状，再看运算

向量是 `(n,)`，矩阵是 `(n, n)`；`A @ v` 保持形状 `(n,)` 不变。本节要验证的关系是 `A @ v == λ * v`：左边是矩阵乘法，右边是标量缩放，两者应逐元素相等。

In [ ]:
import numpy as np
A = np.array([[3,3,3],[3,-1,1],[3,1,-1]], float)
vals, vecs = np.linalg.eig(A)
print('特征值:', sorted(np.round(vals).astype(int)), ' (课件 -3,-2,6)')

## 动手观察：特征向量满足 `A @ v ≈ λ * v`

运行下面的代码，比对 `A @ v` 和 `λ * v` 每个分量是否相等。注意特征值为负时，特征向量方向被翻转；特征值绝对值大于 1 时，向量被拉伸。

In [ ]:
# 验证 Av = λv：遍历 3×3 矩阵的特征向量，计算残差
import numpy as np
A_sym = np.array([[3,3,3],[3,-1,1],[3,1,-1]], float)
vals, vecs = np.linalg.eig(A_sym)
print(f'{"特征值 λ":>12}  {"残差 ‖Av−λv‖":>16}')
for lam, v in zip(vals, vecs.T):
    residual = np.linalg.norm(A_sym @ v - lam * v)
    print(f'λ = {lam:+.4f}  {residual:>16.2e}  {"✅" if residual < 1e-10 else "❌"}')
assert all(
    np.linalg.norm(A_sym @ vecs[:, i] - vals[i] * vecs[:, i]) < 1e-10
    for i in range(len(vals))
), "Av=λv 验证失败：请检查特征向量计算"
print('✅ 所有特征向量满足 Av = λv（残差 < 1e-10）')


In [ ]:
import numpy as np

# 验证特征分解：A = P D P^{-1}
A = np.array([[4., 1.],[2., 3.]])
lam, P = np.linalg.eig(A)
D = np.diag(lam)
A_rec = P @ D @ np.linalg.inv(P)
print(f'特征值 λ = {np.round(lam, 4)}')
print(f'A =\n{A}')
print(f'P@D@P⁻¹ =\n{np.round(A_rec, 10)}')
print(f'重建误差 = {np.max(np.abs(A-A_rec)):.2e}')


## 代码实验：矩阵幂次验证 `Aⁿ = P Dⁿ P⁻¹`

用对角化公式 `Aⁿ = P Dⁿ P⁻¹` 计算矩阵幂次，与 `np.linalg.matrix_power` 结果比较，验证两路计算误差接近机器精度。

In [ ]:
import numpy as np

# 矩阵幂：用对角化 A^n = P D^n P^{-1}
A = np.array([[4., 1.],[2., 3.]])
lam, P = np.linalg.eig(A)
Pinv = np.linalg.inv(P)
for n in [2, 5, 10]:
    A_n_diag = P @ np.diag(lam**n) @ Pinv      # 对角化法 O(N)
    A_n_iter = np.linalg.matrix_power(A, n)     # 直接法（参考）
    err = np.max(np.abs(A_n_diag - A_n_iter))
    print(f'A^{n:2d}[0,0] = {A_n_diag[0,0]:.2f}  误差 = {err:.2e}')


## 2. ✏️ 实现特征多项式（characteristic polynomial）取值 `char_poly(A, lam) = det(A − λI)`

在特征值处它应≈0。

**推理路线**：
1. 构造偏移矩阵 `A - lam * np.eye(len(A))`，把 A 的对角线每个元素减去 λ
2. 在真实特征值 λ 处，该矩阵把特征向量 v 映射到零向量，矩阵因此奇异
3. 奇异矩阵行列式（determinant）恰好为 0；非特征值处行列式明显非零（如 λ=5 时约 56）

**参考输入输出**：A 取课件 3×3 矩阵，`char_poly(A, -3)` ≈ 0，`char_poly(A, 5)` ≈ +56。

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


写 `char_poly` 前明确三件事：
- 输入：矩阵 `A`（n×n）和候选特征值 `lam`（标量）
- 关键步骤：构造 `A - lam * np.eye(n)`，对其求行列式
- 返回：标量，在真实特征值处应接近 0

In [ ]:
def char_poly(A, lam):
    import numpy as np  # guard: 独立运行本 cell 时也能找到 np
    A = np.asarray(A, float)
    # ✏️ TODO: 返回 det(A - lam*I)
    raise NotImplementedError("Implement char_poly: return det(A - lam * np.eye(len(A)))")


In [ ]:
# 还原 3×3 矩阵（cell-6 和 cell-8 中曾将 A 改为 2×2）
import numpy as np
A = np.array([[3,3,3],[3,-1,1],[3,1,-1]], float)

try:
    for lam in (-3, -2, 6):
        print(f'char_poly(A, {lam}) =', round(char_poly(A, lam), 6))
    assert all(abs(char_poly(A, lam)) < 1e-6 for lam in (-3,-2,6)), \
        "char_poly 在特征值处应为 0 —— 请检查 det 的实现"
    assert abs(char_poly(A, 5) - 56) < 1.0, \
        f"char_poly(A, 5) 应 ≈ +56，实际 = {char_poly(A, 5):.2f}"
    print('✅ 特征方程在三个特征值处都≈0，非特征值处返回 +56，与课件一致。')
except NotImplementedError as e:
    print(f'⚠️ 练习尚未实现：{e}')


## 3. 对称矩阵对角化：`PᵀAP = D`（补充例题）

用**归一化**的特征向量按列拼成正交矩阵（orthogonal matrix） P，则 `PᵀAP` = 对角阵(对角线=特征值)。

课件特征向量：$v_1=(1,-1,-1),\ v_2=(0,1,-1),\ v_3=(2,1,1)$。


In [ ]:
A = np.array([[3,3,3],[3,-1,1],[3,1,-1]], float)
v1=np.array([1,-1,-1.]); v2=np.array([0,1,-1.]); v3=np.array([2,1,1.])
P = np.column_stack([v/np.linalg.norm(v) for v in (v1,v2,v3)])
D = P.T @ A @ P
print('P 正交 (PᵀP=I):', np.allclose(P.T@P, np.eye(3)))
print('PᵀAP 对角线:', np.round(np.diag(D)).astype(int), '(课件 -3,-2,6)')
assert np.allclose(D - np.diag(np.diag(D)), 0, atol=1e-9)
print('✅ 成功对角化，与课件一致。')

**🔗 Aurora 连接**：`aurora.audio.transforms` 中的递归滤波器稳定性检验就是对系统矩阵调用 `np.linalg.eig`，确认所有 `|λ| < 1`；PCA 对协方差矩阵做的是同一套特征分解，与 L14 SVD 互为近亲——`AᵀA = VΣ²Vᵀ` 即对 `AᵀA` 做特征分解，特征值恰好是奇异值的平方。

**补充例题对应**：特征方程、对称矩阵性质、正交对角化（orthogonal diagonalization） PᵀAP=D。


## 🎨 图示：对称矩阵的正交对角化 S = QΛQᵀ (补充例题)


In [ ]:
import numpy as np
from aurora.laviz import style, show_factorization
style()
S=np.array([[3.,3,3],[3,-1,1],[3,1,-1]]); w,Q=np.linalg.eigh(S)
show_factorization(S,[Q,np.diag(w),Q.T],['Q','Λ','Qᵀ'],
                   modes=['#2A9D8F','#E9C46A','#2A9D8F'], title='S = Q Λ Qᵀ');

In [ ]:
import numpy as np

# 参数实验：改变矩阵参数观察特征值变化
print(f'{'参数 c':>8}  {'λ₁':>10}  {'λ₂':>10}  {'|λ₁/λ₂|':>10}')
for c in [0.5, 1.0, 2.0, 4.0]:
    A = np.array([[3., c],[1., 2.]])
    lam = np.linalg.eigvals(A)
    lam = sorted(lam, key=abs, reverse=True)
    ratio = abs(lam[0]/lam[1]) if abs(lam[1]) > 1e-10 else float('inf')
    print(f'{c:>8.1f}  {lam[0]:>+10.4f}  {lam[1]:>+10.4f}  {ratio:>10.4f}')
print('→ 两特征值相差越大，幂迭代收敛越快（主导方向越突出）。')


## 参数实验：稳定边界的两侧

构造特征值分别为 0.9 和 1.1 的矩阵，计算 A^50，观察两个方向的长期行为：
- `0.9^50 ≈ 0.005`：该特征方向已衰减到接近零（稳定）
- `1.1^50 ≈ 117`：该特征方向指数爆炸（不稳定）

仅凭特征值模是否越过 1.0 这条线，就能预判系统长期行为。Aurora 状态空间模型的稳定性保证就是把系统矩阵所有特征值的模约束在单位圆内。

## 本课收束

现在可以用 `np.linalg.eig(A)` 取得特征值数组和特征向量矩阵 `P`，并用 `char_poly(A, lam)` 验证任意 `λ` 是否为真正的特征值。这对应 Aurora 状态空间模型的稳定性检验：把系统矩阵传入特征分解，若所有 `|λ| < 1`，递归滤波器就收敛。下一节（**L18**）补全可逆性判据，使 `A = PΛP⁻¹` 可以精确还原。

## ✏️ 白板挑战：特征分解手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：A = [[4, 1], [2, 3]]，手算特征方程 `det(A - λI) = 0`，求两个特征值。

**问 2**：验证：对问 1 的较大特征值 λ₁，`char_poly(A, λ₁)` 应等于多少？

**问 3**：若 A = P diag(2, 5) P⁻¹，A³ = P diag(?, ?) P⁻¹？（不用知道 P 是什么）

**问 4**：一个线性系统的系统矩阵特征值为 [0.9, −0.8]，这个系统稳定吗？  
（判据：所有 |λᵢ| < 1 则稳定）

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

A = np.array([[4., 1.], [2., 3.]])

# 问1：特征值
vals_ref = np.sort(np.linalg.eigvals(A))
assert np.allclose(np.sort(vals_ref), [2., 5.], atol=1e-10)
print(f"Q1 ✅  A=[[4,1],[2,3]] 特征值 = {np.sort(vals_ref)}  (手算: λ²-7λ+10=0 → 5,2)")

# 问2：char_poly(A, λ₁) ≈ 0
lam1 = 5.0
try:
    cp = char_poly(A, lam1)
    assert abs(cp) < 1e-8, f"char_poly(A, 5) 应≈0，得到 {cp}"
    print(f"Q2 ✅  char_poly(A, 5) = {cp:.2e}  （λ=5 是真正的特征值）")
except NotImplementedError:
    cp_manual = np.linalg.det(A - lam1 * np.eye(2))
    print(f"Q2 ✅  det(A-5I) = {cp_manual:.2e} ≈ 0  (char_poly 待实现)")

# 问3：A³ = P diag(2³,5³) P⁻¹
lam3_expected = [2**3, 5**3]  # = [8, 125]
vals3, vecs3 = np.linalg.eig(A)
D3 = np.diag(vals3**3)
A3_diag = vecs3 @ D3 @ np.linalg.inv(vecs3)
A3_ref  = np.linalg.matrix_power(A.astype(int), 3).astype(float)
assert np.allclose(A3_diag, A3_ref, atol=1e-8)
print(f"Q3 ✅  A³ = P·diag({lam3_expected[0]},{lam3_expected[1]})·P⁻¹  (=diag(2³,5³))")

# 问4：稳定性（所有|λ|<1）
eigs_sys = [0.9, -0.8]
stable = all(abs(e) < 1 for e in eigs_sys)
assert stable
print(f"Q4 ✅  特征值={eigs_sys}，|0.9|<1 且 |-0.8|<1 → 系统稳定（Aⁿ→0）")
print("\n🎉 特征分解白板挑战通过！换坐标系让乘法变简单的思想已内化。")

In [ ]:
# ✏️ 本课自评
l17_review = {
    "char_poly_implemented":       None,  # char_poly 实现并通过断言？True/False
    "eigendecomp_A_eq_PDP_inv":   None,  # 理解 A=PDP⁻¹ 的含义？True/False
    "matrix_power_via_diag":      None,  # 知道 Aⁿ=PDⁿP⁻¹？True/False
    "stability_criterion":         None,  # 理解|λ|<1→稳定？True/False
    "whiteboard_passed":           None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l17_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l17_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L17 全部通关！进入 L18：可逆性与秩')

---

→ **下一课**　[L18 · 可逆性与秩](L18_invertibility.ipynb)

> 下节课将学习 **可逆性与秩**：秩 = 信息量，零空间 = 被压缩的方向，奇异矩阵诊断。